In [5]:
import os
import pytesseract
from PIL import Image, ImageOps, ImageEnhance
import pandas as pd
import numpy as np
from pptx import Presentation
from io import BytesIO
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils.dataframe import dataframe_to_rows
from unstructured.partition.pptx import partition_pptx
import cv2
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment

pytesseract.pytesseract.tesseract_cmd = r'C:\Users\RW565TZ\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

In [6]:
class PPTExtractor:
    def __init__(self):
        # Replace EasyOCR with pytesseract
        self.workbook = self.create_workbook()

    def create_workbook(self):
        wb = Workbook()
        return wb

    def format_text(self, text):
        if text is None:
            return ""
        forbidden_chars = ['\x00', '\x0B', '\x0C', '\x0E', '\x0F']    
        for char in forbidden_chars:
            text = text.replace(char, '')
        return text

    def write(self, sheet, content_type, content, row_num):
        content = self.format_text(content)
        cell = sheet.cell(row=row_num, column=1)
        cell.value = f"{content_type}:"
        cell.alignment = Alignment(wrap_text=True)
        
        content_cell = sheet.cell(row=row_num, column=2)
        content_cell.value = content
        content_cell.alignment = Alignment(wrap_text=True)
        
        return row_num + 1

    def extract_content(self, ppt_path, slide_numbers):
        import pytesseract
        from PIL import Image
        import numpy as np
        import io
        
        from openpyxl import Workbook
        from openpyxl.styles import Alignment
        
        prs = Presentation(ppt_path)
        slide_indices = [num - 1 for num in slide_numbers]
        
        extracted_content = {}
        
        for slide_idx in slide_indices:
            if slide_idx >= len(prs.slides):
                print(f"Warning: Slide {slide_idx + 1} does not exist in the presentation")
                continue
                
            slide = prs.slides[slide_idx]
            sheet_name = f"Slide_{slide_idx + 1}"

            sheet = self.workbook.create_sheet(sheet_name)
            sheet.column_dimensions['A'].width = 15
            sheet.column_dimensions['B'].width = 100
            
            current_row = 1
            
            slide_content = {
                'text': [],
                'table': [],
                'image_text': []
            }
            
            sheet.cell(row=current_row, column=1).value = f"Slide {slide_idx + 1}"
            current_row += 2
            
            for shape in slide.shapes:
                if shape.has_text_frame:
                    text_content = []
                    for paragraph in shape.text_frame.paragraphs:
                        if paragraph.text.strip():
                            text_content.append(paragraph.text.strip())
                    
                    if text_content:
                        slide_content['text'].extend(text_content)
                        text_to_write = "\n".join(text_content)
                        current_row = self.write(sheet, "Text", text_to_write, current_row)
                        current_row += 1

                elif shape.has_table:
                    table_data = []
                    for row in shape.table.rows:
                        row_data = []
                        for cell in row.cells:
                            cell_text = cell.text.strip()
                            if cell_text:
                                row_data.append(cell_text)
                        if row_data:
                            table_data.append(row_data)
                    
                    if table_data:
                        slide_content['table'].append(table_data)
                        table_text = "\n".join(" | ".join(str(cell) for cell in row) for row in table_data)
                        current_row = self.write(sheet, "Table", table_text, current_row)
                        current_row += 1

                elif hasattr(shape, 'image'):
                    try:
                        
                        image_stream = io.BytesIO(shape.image.blob)
                        image = Image.open(image_stream)
                        
                        
                        image_text = pytesseract.image_to_string(
                            image
                        )
                        
                        if image_text.strip():
                            image_text_lines = [line.strip() for line in image_text.split('\n') if line.strip()]
                            
                            if image_text_lines:
                                slide_content['image_text'].extend(image_text_lines)
                                image_text_to_write = "\n".join(image_text_lines)
                                current_row = self.write(sheet, "Image Text", image_text_to_write, current_row)
                                current_row += 1
                    except Exception as e:
                        print(f"Error processing image in slide {slide_idx + 1}: {str(e)}")

            extracted_content[slide_idx + 1] = slide_content
        
        output_file = "Extracted_PPTcontent.xlsx"
        self.workbook.save(output_file)
        return extracted_content

In [ ]:
def main():
    ppt_path = r"C:\Users\RW565TZ\Downloads\Cost Diagnostics Report_Draft 26_9_24.pptx"
    slide_numbers = [7,8,9,12,15,43,48,49,50,51,52,53,54,55,56,57,58,59,60,61,63,64,65,66]
    
    try:
        extractor = PPTExtractor()
        extracted_content = extractor.extract_content(ppt_path, slide_numbers)
       
        print(f"\nProcess Complete")
    except Exception as e:
        print(f"Error processing presentation: {str(e)}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import comtypes.client

def convert_ppt_to_pdf(input_ppt, output_pdf):
    powerpoint = comtypes.client.CreateObject("PowerPoint.Application")
    powerpoint.Visible = 1  

    ppt = powerpoint.Presentations.Open(input_ppt, WithWindow=False)
    ppt.SaveAs(output_pdf, 32) 
    ppt.Close()
    
    powerpoint.Quit()

# Example usage
input_ppt = r"C:\Users\Downloads\Report_Draft.pptx"
output_pdf = r"C:\Users\Downloads\Report_Draft.pdf"
convert_ppt_to_pdf(input_ppt, output_pdf)


In [ ]:
import io
import pytesseract
from PIL import Image
from pptx import Presentation
from unstructured.partition.pptx import partition_pptx
from openpyxl import Workbook
from openpyxl.styles import Alignment

class PPTExtractor:
    def __init__(self):
        self.workbook = self.create_workbook()

    def create_workbook(self):
        return Workbook()

    def format_text(self, text):
        if text is None:
            return ""
        forbidden_chars = ['\x00', '\x0B', '\x0C', '\x0E', '\x0F']
        for char in forbidden_chars:
            text = text.replace(char, '')
        return text

    def write(self, sheet, content_type, content, row_num):
        content = self.format_text(content)
        sheet.cell(row=row_num, column=1).value = f"{content_type}:"
        sheet.cell(row=row_num, column=2).value = content
        sheet.cell(row=row_num, column=2).alignment = Alignment(wrap_text=True)
        return row_num + 1

    def extract_text_from_images(self, shape):
        """Extracts text from images using Tesseract OCR."""
        try:
            if hasattr(shape, "image") and shape.image:
                image_stream = io.BytesIO(shape.image.blob)
                image = Image.open(image_stream)

                # Convert image to text using OCR
                extracted_text = pytesseract.image_to_string(image)
                return extracted_text.strip()
        except Exception as e:
            print(f"Error extracting text from image: {str(e)}")
        return ""

    def extract_content(self, ppt_path, slide_numbers):
        elements = partition_pptx(filename=ppt_path)
        prs = Presentation(ppt_path)
        extracted_content = {}

        for slide_num in slide_numbers:
            sheet_name = f"Slide_{slide_num}"
            sheet = self.workbook.create_sheet(sheet_name)
            sheet.column_dimensions['A'].width = 15
            sheet.column_dimensions['B'].width = 100
            current_row = 1

            # Extract elements using unstructured
            slide_elements = [e for e in elements if e.metadata.page_number == slide_num]
            slide_content = {"text": [], "table": [], "image_text": []}

            if not slide_elements:
                print(f"Warning: No content extracted from slide {slide_num}")

            for element in slide_elements:
                if element.category == "Text":
                    slide_content["text"].append(element.text)
                    current_row = self.write(sheet, "Text", element.text, current_row)

                elif element.category == "Table":
                    table_rows = [row.split("\t") for row in element.text.split("\n")]
                    table_text = "\n".join([" | ".join(row) for row in table_rows])

                    slide_content["table"].append(table_text)
                    current_row = self.write(sheet, "Table", table_text, current_row)

            # Extract images & OCR text manually from PPTX
            slide_index = slide_num - 1
            if slide_index < len(prs.slides):
                slide = prs.slides[slide_index]
                for shape in slide.shapes:
                    image_text = self.extract_text_from_images(shape)
                    if image_text:
                        slide_content["image_text"].append(image_text)
                        current_row = self.write(sheet, "Image Text", image_text, current_row)

            extracted_content[slide_num] = slide_content

        self.workbook.save("Extracted_PPTcontent.xlsx")
        return extracted_content

def main():
    ppt_path = r"C:\Users\Downloads\Report_Draft.pptx"
    slide_numbers = [7, 8, 9, 12, 15, 43, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66]

    try:
        extractor = PPTExtractor()
        extractor.extract_content(ppt_path, slide_numbers)
        print("\nProcess Complete")
    except Exception as e:
        print(f"Error processing PPT: {str(e)}")

if __name__ == "__main__":
    main()


In [ ]:
import os
import io
from openpyxl import Workbook
from openpyxl.styles import Alignment
from pptx import Presentation
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract
from langchain.chat_models import AzureChatOpenAI

llm = AzureChatOpenAI(
    openai_api_key="",
    openai_api_version="",
    azure_endpoint="",
    deployment_name='',
    model_name="gpt-4o-mini",
    temperature=0.1,
    max_tokens=2000
)

class PPTExtractor:
    def __init__(self):
        self.workbook = Workbook()

    def format_text(self, text):
        if text is None:
            return ""
        forbidden_chars = ['\x00', '\x0B', '\x0C', '\x0E', '\x0F']    
        for char in forbidden_chars:
            text = text.replace(char, '')
        return text

    def write(self, sheet, content_type, content, row_num):
        content = self.format_text(content)
        cell = sheet.cell(row=row_num, column=1)
        cell.value = f"{content_type}:"
        cell.alignment = Alignment(wrap_text=True)
        
        content_cell = sheet.cell(row=row_num, column=2)
        content_cell.value = content
        content_cell.alignment = Alignment(wrap_text=True)
        
        return row_num + 1

    def process_with_gpt(self, content, content_type):
        if not content.strip():
            return ""
        
        prompt = f"Extract the most relevant numerical values and key insights from the following {content_type} in a PowerPoint slide:\n\n{content}\n\nProvide only key values and insights, ignoring axes labels and scales."
        
        response = llm.predict(prompt)
        return response.strip()

    def extract_content(self, ppt_path, slide_numbers):
        prs = Presentation(ppt_path)
        slide_indices = [num - 1 for num in slide_numbers]
        extracted_content = {}
        
        for slide_idx in slide_indices:
            if slide_idx >= len(prs.slides):
                print(f"Warning: Slide {slide_idx + 1} does not exist in the presentation")
                continue
                
            slide = prs.slides[slide_idx]
            sheet_name = f"Slide_{slide_idx + 1}"
            sheet = self.workbook.create_sheet(sheet_name)
            sheet.column_dimensions['A'].width = 15
            sheet.column_dimensions['B'].width = 100
            current_row = 1
            
            slide_content = {'text': [], 'table': [], 'image_text': []}
            sheet.cell(row=current_row, column=1).value = f"Slide {slide_idx + 1}"
            current_row += 2
            
            for shape in slide.shapes:
                if shape.has_text_frame:
                    text_content = "\n".join([para.text.strip() for para in shape.text_frame.paragraphs if para.text.strip()])
                    if text_content:
                        slide_content['text'].append(text_content)
                        current_row = self.write(sheet, "Text", text_content, current_row)
                        current_row += 1

                elif shape.has_table:
                    table_data = [
                        " | ".join([cell.text.strip() for cell in row.cells if cell.text.strip()])
                        for row in shape.table.rows if any(cell.text.strip() for cell in row.cells)
                    ]
                    
                    if table_data:
                        table_text = "\n".join(table_data)
                        slide_content['table'].append(table_text)
                        current_row = self.write(sheet, "Table", table_text, current_row)
                        current_row += 1

                elif hasattr(shape, 'image'):
                    try:
                        image_stream = io.BytesIO(shape.image.blob)
                        image = Image.open(image_stream)
                        image = image.convert("L").filter(ImageFilter.MedianFilter())
                        enhancer = ImageEnhance.Contrast(image)
                        image = enhancer.enhance(2)
                        image_text = pytesseract.image_to_string(image)
                        
                        if image_text.strip():
                            relevant_image_text = self.process_with_gpt(image_text, "image text")
                            slide_content['image_text'].append(relevant_image_text)
                            current_row = self.write(sheet, "Image Text", relevant_image_text, current_row)
                            current_row += 1
                    except Exception as e:
                        print(f"Error processing image in slide {slide_idx + 1}: {str(e)}")
            
            extracted_content[slide_idx + 1] = slide_content
        
        output_file = "Extracted_PPTcontent.xlsx"
        self.workbook.save(output_file)
        return extracted_content

def main():
    ppt_path = r"C:\\Users\\Downloads\\Report_Draft.pptx"
    slide_numbers = [7,8,9,12,15,43,48,49,50,51,52,53,54,55,56,57,58,59,60,61,63,64,65,66]
    
    try:
        extractor = PPTExtractor()
        extracted_content = extractor.extract_content(ppt_path, slide_numbers)
       
        print(f"\nProcess Complete")
    except Exception as e:
        print(f"Error processing presentation: {str(e)}")

if __name__ == "__main__":
    main()

In [ ]:
import io
import pytesseract
import base64
from PIL import Image
from pptx import Presentation
from unstructured.partition.pptx import partition_pptx
from openpyxl import Workbook
from openpyxl.styles import Alignment
from langchain.chat_models.azure_openai import AzureChatOpenAI

class PPTExtractor:
    def __init__(self, azure_client=None):
        self.workbook = self.create_workbook()
        self.azure_client = azure_client

    def create_workbook(self):
        return Workbook()

    def format_text(self, text):
        if text is None:
            return ""
        forbidden_chars = ['\x00', '\x0B', '\x0C', '\x0E', '\x0F']
        for char in forbidden_chars:
            text = text.replace(char, '')
        return text

    def write(self, sheet, content_type, content, row_num):
        content = self.format_text(content)
        sheet.cell(row=row_num, column=1).value = f"{content_type}:"
        sheet.cell(row=row_num, column=2).value = content
        sheet.cell(row=row_num, column=2).alignment = Alignment(wrap_text=True)
        return row_num + 1

    def extract_text_from_images(self, shape):
    
        try:
            if hasattr(shape, "image") and shape.image:
                image_stream = io.BytesIO(shape.image.blob)
                image = Image.open(image_stream)

                extracted_text = pytesseract.image_to_string(image)
                return extracted_text.strip()
        except Exception as e:
            print(f"Error extracting text from image: {str(e)}")
        return ""

    def analyze_image_with_openai(self, image_blob):
 
        if not self.azure_client:
            return "OpenAI client not configured."
        
        try:
            image_base64 = base64.b64encode(image_blob).decode('utf-8')
            
            response = self.azure_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system", 
                        "content": "You are an expert at analyzing visual content from PowerPoint slides. Provide detailed analysis of any tables, charts, or graphs in the image. For tables: extract the structured data and key insights. For charts/graphs: describe the type of visualization, what data is being presented, the trends shown, and the main conclusions that can be drawn. Be factual and detailed."
                    },
                    {
                        "role": "user", 
                        "content": [
                            {"type": "text", "text": "Analyze this image from a PowerPoint slide. Describe what you see, extract any data, and share key insights."},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                        ]
                    }
                ],
                temperature=0.1,
                max_tokens=2000
            )
            
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            print(f"Error analyzing image with OpenAI: {str(e)}")
            return f"Error during image analysis: {str(e)}"

    def extract_content(self, ppt_path, slide_numbers):
        elements = partition_pptx(filename=ppt_path)
        prs = Presentation(ppt_path)
        extracted_content = {}

        for slide_num in slide_numbers:
            sheet_name = f"Slide_{slide_num}"
            sheet = self.workbook.create_sheet(sheet_name)
            sheet.column_dimensions['A'].width = 15
            sheet.column_dimensions['B'].width = 100
            current_row = 1
            
            slide_elements = [e for e in elements if e.metadata.page_number == slide_num]
            slide_content = {"text": [], "table": [], "image_text": [], "ai_image_analysis": []}

            if not slide_elements:
                print(f"Warning: No content extracted from slide {slide_num}")

            for element in slide_elements:
                if element.category == "Text":
                    slide_content["text"].append(element.text)
                    current_row = self.write(sheet, "Text", element.text, current_row)

                elif element.category == "Table":
                    table_rows = [row.split("\t") for row in element.text.split("\n")]
                    table_text = "\n".join([" | ".join(row) for row in table_rows])

                    slide_content["table"].append(table_text)
                    current_row = self.write(sheet, "Table", table_text, current_row)

            slide_index = slide_num - 1
            if slide_index < len(prs.slides):
                slide = prs.slides[slide_index]
                for shape in slide.shapes:
                    if hasattr(shape, "image") and shape.image:
               
                        image_text = self.extract_text_from_images(shape)
                        if image_text:
                            slide_content["image_text"].append(image_text)
                            current_row = self.write(sheet, "OCR Text", image_text, current_row)
                        
                        if self.azure_client:
                            ai_analysis = self.analyze_image_with_openai(shape.image.blob)
                            slide_content["ai_image_analysis"].append(ai_analysis)
                            current_row = self.write(sheet, "AI Analysis", ai_analysis, current_row)

            extracted_content[slide_num] = slide_content

        if "Sheet" in self.workbook.sheetnames:
            self.workbook.remove(self.workbook["Sheet"])
            
        self.workbook.save("Extracted_PPTcontent.xlsx")
        return extracted_content

def main():
    ppt_path = r"C:\Users\Downloads\Report_Draft.pptx"
    slide_numbers = [9,60, 61, 63, 64, 65, 66]

    try:
        azure_client = AzureChatOpenAI(
            openai_api_key="",
            openai_api_version="",
            azure_endpoint="",
            deployment_name='',
            model_name="gpt-4o-mini",
            temperature=0.1,
            max_tokens=2000
        )
        
        extractor = PPTExtractor(azure_client=azure_client)
        extractor.extract_content(ppt_path, slide_numbers)
        print("\nProcess Complete")
    except Exception as e:
        print(f"Error processing PPT: {str(e)}")

if __name__ == "__main__":
    main()